In [ ]:
# Phase 1: ingest one BTS Marketing Carrier On-Time Performance month into lh_bronze.
# Downloads the monthly zip, lands the raw CSV into Files/bts/yyyy/mm/ for
# traceability, then writes it to a bronze Delta table with all columns as
# strings plus _source_file and _loaded_at, per contracts/bts_marketing_ontime.yaml.

import os
import urllib.request
import zipfile
from datetime import datetime, timezone

YEAR, MONTH = 2026, 8
WORKSPACE_ID = "16b2a9c6-6418-485f-865f-c47442b8b6c6"
LH_BRONZE_ID = "09a8b1f5-f33c-4eed-912d-40b9cbdc9b06"
ABFSS_ROOT = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{LH_BRONZE_ID}"

URL = (
    "https://transtats.bts.gov/PREZIP/On_Time_Marketing_Carrier_On_Time_Performance_"
    f"Beginning_January_2018_{YEAR}_{MONTH}.zip"
)
LOCAL_ZIP = f"/tmp/bts_{YEAR}_{MONTH}.zip"
LOCAL_DIR = f"/tmp/bts_{YEAR}_{MONTH}"

print(f"downloading {URL}")
urllib.request.urlretrieve(URL, LOCAL_ZIP)
print(f"downloaded {os.path.getsize(LOCAL_ZIP):,} bytes")


In [ ]:
os.makedirs(LOCAL_DIR, exist_ok=True)
with zipfile.ZipFile(LOCAL_ZIP) as z:
    z.extractall(LOCAL_DIR)
csv_files = [f for f in os.listdir(LOCAL_DIR) if f.endswith(".csv")]
print("extracted:", csv_files)
local_csv_path = os.path.join(LOCAL_DIR, csv_files[0])


In [ ]:
# land the raw CSV into the lakehouse Files area (matches planning/project-plan.md's
# Files/bts/{yyyy}/{mm}/ landing pattern) before any transformation, for traceability
files_dest_dir = f"{ABFSS_ROOT}/Files/bts/{YEAR}/{MONTH:02d}"
csv_dest_path = f"{files_dest_dir}/{csv_files[0]}"
notebookutils.fs.cp(f"file:{local_csv_path}", csv_dest_path)
print(f"landed to {csv_dest_path}")


In [ ]:
from pyspark.sql.functions import col, current_timestamp, lit

df = spark.read.option("header", True).csv(csv_dest_path)

# Delta rejects column names containing ' ,;{}()\n\t=' -- the real BTS file
# has one (see contracts/bts_marketing_ontime.yaml): "Operating_Airline "
# has a trailing space. Strip whitespace and swap any other invalid char
# for '_' rather than hardcoding the one known offender, in case BTS
# introduces another in a future month.
import re

INVALID_DELTA_CHARS = re.compile(r'[ ,;{}()\n\t=]')


def sanitize_column_name(name: str) -> str:
    return INVALID_DELTA_CHARS.sub("_", name.strip())


rename_map = {c: sanitize_column_name(c) for c in df.columns}
changed = {k: v for k, v in rename_map.items() if k != v}
if changed:
    print(f"sanitizing {len(changed)} column name(s): {changed}")
df = df.select([col(c).alias(rename_map[c]) for c in df.columns])

# bronze contract: every column as string, plus lineage columns
df = df.select([col(c).cast("string").alias(c) for c in df.columns])
df = (
    df.withColumn("_source_file", lit(csv_files[0]))
    .withColumn("_loaded_at", current_timestamp())
)

row_count = df.count()
print(f"rows: {row_count:,}, columns: {len(df.columns)}")


In [ ]:
bronze_table_path = f"{ABFSS_ROOT}/Tables/bts_marketing_ontime"
df.write.format("delta").mode("overwrite").save(bronze_table_path)
print(f"wrote bronze Delta table to {bronze_table_path}")


In [ ]:
# verification -- AA rows at DFW should be present and match what the
# local pipeline validated against the same real file (see
# tests/test_bts_times_real_data.py, 13,332 AA/DFW legs for this month)
check = spark.read.format("delta").load(bronze_table_path)
total = check.count()
aa_dfw = check.filter(
    (col("Marketing_Airline_Network") == "AA")
    & ((col("Origin") == "DFW") | (col("Dest") == "DFW"))
).count()
print(f"bronze table total rows: {total:,}")
print(f"AA rows touching DFW: {aa_dfw:,}")
